In [5]:
import time
import requests
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from io import StringIO
from tqdm import tqdm
from numpy import nan
import pandas as pd

url = requests.get("https://finance.naver.com/sise/sise_market_sum.naver?sosok=0&page=1")
url.text
# html = BeautifulSoup(url.text, 'html.parser')
html = BeautifulSoup(url.text)
html
table = html.find('table', {'class':'type_2'})
table

table_str = str(table)
table_io = StringIO(table_str)

tables = pd.read_html(table_io)[0]
tables = tables[tables['종목명'].notnull()]

# tables = tables.drop(['N', '토론실'], axis=1)
tables = tables.drop(['N', '토론'], axis=1)
tables.head()

kospi_box = []
for page in tqdm(range(1, 50)):
    url = requests.get(f"https://finance.naver.com/sise/sise_market_sum.naver?sosok=0&page={page}")
    html = BeautifulSoup(url.text)
    
    table = html.find('table', {'class':'type_2'})
    table_str = str(table)
    table_io = StringIO(table_str)

    tables = pd.read_html(table_io)[0]
    tables = tables[tables['종목명'].notnull()]
    
    tables = tables.drop(['N', '토론'], axis=1)
    tables['소속'] = 'KOSPI'
    kospi_box.append(tables)
    time.sleep(1)

kosdaq_box = []
for page in tqdm(range(1, 40)):
    url = requests.get(f"https://finance.naver.com/sise/sise_market_sum.naver?sosok=1&page={page}")
    html = BeautifulSoup(url.text)
    
    table = html.find('table', {'class':'type_2'})
    table_str = str(table)
    table_io = StringIO(table_str)

    tables = pd.read_html(table_io)[0]
    tables = tables[tables['종목명'].notnull()]
    
    tables = tables.drop(['N', '토론'], axis=1)
    tables['소속'] = 'KOSDAQ'
    kosdaq_box.append(tables)
    time.sleep(1)

# stock = pd.concat(kospi_box + kosdaq_box, axis=0)
stock = pd.concat(kospi_box + kosdaq_box, ignore_index=True )
stock

sample = stock.dropna(subset='PER')
sample = sample[(sample['PER'] > 0) & (sample['PER'] < 10)]
sample = sample.sort_values(by='PER')

for i in range(len(sample)):

    data = sample.iloc[i]
    
    name = data['종목명']
    per = data['PER']

    # print(f"{i+1:3d} {name:15s} {per:6.2f}")
    print (f"{name:15s} PER : {per:6.2f}")

sample = stock.dropna(subset='ROE')
sample = sample[sample['ROE']>0]
sample = sample.sort_values("ROE", ascending=False)

for i in range(len(sample)):

    data = sample.iloc[i]
    
    name = data['종목명']
    roe = data['ROE']

    print (f"{name:15s} ROE : {roe:6.2f}")





100%|██████████| 39/39 [00:55<00:00,  1.41s/it]


오션인더블유          PER :   0.36
효성화학            PER :   0.41
한창              PER :   0.47
예림당             PER :   0.56
DMS             PER :   0.70
DH오토넥스          PER :   1.16
씨엑스아이           PER :   1.35
SB성보            PER :   1.36
KC코트렐           PER :   1.66
일정실업            PER :   1.72
서전기전            PER :   1.78
웅진              PER :   1.87
포니링크            PER :   1.87
옵트론텍            PER :   1.90
우성              PER :   1.91
서한              PER :   1.93
위메이드플레이         PER :   1.95
전방              PER :   1.98
베셀              PER :   2.03
한진중공업홀딩스        PER :   2.05
일동홀딩스           PER :   2.16
케이비아이동국실업       PER :   2.20
코오롱글로벌          PER :   2.25
유성티엔에스          PER :   2.26
SG&G            PER :   2.29
현대해상            PER :   2.29
우원개발            PER :   2.32
솔본              PER :   2.33
대원산업            PER :   2.33
계룡건설            PER :   2.37
DSR제강           PER :   2.39
넥센타이어1우B        PER :   2.42
에코캡             PER :   2.42
현대지에프홀딩스        PER :   2.45
기산텔레콤         

In [16]:
from bs4 import BeautifulSoup
from io import StringIO
import requests
import pandas as pd
import time
from tqdm import tqdm

url = requests.get("https://finance.naver.com/sise/dividend_list.naver?&page=1")
html = BeautifulSoup(url.text)

# table = html.find('table', {'class':'type_2'})

table = html.find('table', class_='type_1 tb_ty')
table = StringIO(str(table))
table = pd.read_html(table, header = 1)[0]
table.dropna (subset = '종목명', inplace = True)
table 

total = []
for page in tqdm(range(1, 10)):
    
    url = requests.get(f"https://finance.naver.com/sise/dividend_list.naver?&page={page}")
    html = BeautifulSoup(url.text)

    table = html.find('table', class_='type_1 tb_ty')
    table = StringIO(str(table))
    table = pd.read_html(table, header = 1)[0]
    table.dropna (subset = '종목명', inplace = True)
    total.append(table)
    time.sleep(1)

total_table = pd.concat(total, ignore_index=True)
total_table

total_table.sort_values('수익률 (%)', ascending=False, inplace=True)
total_table.sort_values('수익률 (%)', ascending=False).head(10)

total_table = total_table.replace('-', nan)
total_table = total_table.dropna(subset=['배당금', '1년전', '2년전', '3년전'])

total_table['배당금'] = total_table['배당금'].astype(int)
total_table['1년전'] = total_table['1년전'].astype(int)
total_table['2년전'] = total_table['2년전'].astype(int)
total_table['3년전'] = total_table['3년전'].astype(int)

A=total_table['3년전'] < total_table['2년전']
B=total_table['2년전'] < total_table['1년전']
C=total_table['1년전'] < total_table['배당금']

# total_table [A & B & C]
total_table [A & B & C].sort_values('수익률 (%)', ascending=False).head(10)


100%|██████████| 9/9 [00:12<00:00,  1.38s/it]
/var/folders/l3/299twkq15zb915fh0byg52qc0000gn/T/ipykernel_2691/2053204330.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  total_table = pd.concat(total, ignore_index=True)


,종목명,현재가,기준월,배당금,수익률 (%),배당성향 (%),ROE (%),PER (배),PBR (배),1년전,2년전,3년전
3,NH프라임리츠,4350.0,25.05,644,14.81,21.84,9.88,7.79,0.75,540,246,233
18,이지스레지던스리츠,3895.0,25.06,300,7.70,159.24,1.18,43.68,0.49,278,262,261
21,한솔로지스틱스,2730.0,25.12,200,7.33,NaN,NaN,NaN,NaN,150,100,70
29,HS애드,8270.0,25.03,550,6.65,39.29,11.77,4.63,0.51,450,400,350
30,강원랜드,17680.0,25.04,1170,6.62,51.32,12.08,7.48,0.82,930,350,0
38,KG이니시스,9710.0,25.12,600,6.18,NaN,NaN,NaN,NaN,500,420,400
58,DB손해보험,124500.0,25.03,6800,5.46,22.05,18.98,3.93,0.66,5300,4600,3500
62,무림페이퍼,1884.0,25.03,100,5.31,10.23,9.39,2.10,0.19,75,50,25
66,기업은행,20375.0,25.03,1065,5.23,32.11,8.06,4.32,0.34,984,960,780
68,삼성화재우,365000.0,25.03,19005,5.21,38.95,13.11,8.74,0.98,16005,13805,12005
